In [ ]:
# Data Manipulation with Pandas

Pandas is the most widely used library of python for data science. It is incredibly helpful in manipulating the data so that you can derive better insights and build great machine learning models.

In this notebook, we will have a look at some of the intermediate concepts of working with pandas.


## Table of Contents

 - Apply function

### Loading dataset

***In this notebook we will use the Big Mart Sales Data. You can download the data from : https://datahack.analyticsvidhya.com/contest/practice-problem-big-mart-sales-iii/download/train-file*** 
import pandas as pd
import numpy as np

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")
# reset index after dropping
data_BM = data_BM.reset_index(drop=True)
# view the top results
data_BM.head()
### Apply function

- Apply function can be used to perform pre-processing/data-manipulation on your data both row wise and column wise.
- It is a faster method than simply using a **for** loop over your dataframe.
- Almost every time I need to iterate over a dataframe or it's rows/columns, I will think of using the `apply`.
- Hence, it is widely used in feature engineering code.
# accessing row wise
data_BM.apply(lambda x: x)
# access first row
data_BM.apply(lambda x: x[0])
# accessing column wise
data_BM.apply(lambda x: x, axis=1)
# access first column by index
data_BM.apply(lambda x: x[0], axis=1)
# access by column name
data_BM.apply(lambda x: x["Item_Fat_Content"], axis=1)
- You can also use `apply` to implement a **condition** individually on every row/column of your dataframe.
- Suppose you want to clip Item_MRP to 200 and not consider any value greater than that.
```python
def clip_price(price):
    if price > 200:
        price = 200
    return price
```
# before clipping
data_BM["Item_MRP"][:5]
# clip price if it is greater than 200
def clip_price(price):
    if price > 200:
        price = 200
    return price

# after clipping
data_BM["Item_MRP"].apply(lambda x: clip_price(x))[:5]
- Suppose you want to label encode Outlet_Location_Type as 0, 1 and 2 for Tier 1, Tier 2 and Tier 3 city, your logic would be:

```python
def label_encode(city):
    if city == 'Tier 1':
        label = 0
    elif city == 'Tier 2':
        label = 1
    else:
        label = 2
    return label
```
- You can use the `apply` to operate `label_encode` logic on every row of the Outlet_Location_Type column.
# before label encoding
data_BM["Outlet_Location_Type"][:5]
# label encode city type
def label_encode(city):
    if city == 'Tier 1':
        label = 0
    elif city == 'Tier 2':
        label = 1
    else:
        label = 2
    return label

# operate label_encode on every row of Outlet_Location_Type
data_BM["Outlet_Location_Type"] = data_BM["Outlet_Location_Type"].apply(label_encode)
# after label encoding
data_BM["Outlet_Location_Type"][:5]

In [ ]:
# Data Manipulation with Pandas

### Loading dataset

***In this notebook we will use the Big Mart Sales Data. You can download the data from : https://datahack.analyticsvidhya.com/contest/practice-problem-big-mart-sales-iii/download/train-file*** 
import pandas as pd
import numpy as np

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")
# reset index after dropping
data_BM = data_BM.reset_index(drop=True)
# view the top results
data_BM.head()
### 1. Aggregating data

There are multiple functions that can be used to perform useful aggregations on data in pandas:

 - groupby
 - crosstab
 - pivottable
 
#### a. **What is the mean price for each item type?** : groupby
- In the given data set, I want to find out **what is the mean price for each item type**?
- You can use **groupby()** to achieve this.
- The first step would be to group the data by Item_Type column.
# group price based on item type
price_by_item = data_BM.groupby('Item_Type')

# display first few rows
price_by_item.first()
- Now that you have grouped by Item_Type, the next step would be to calculate the mean of Item_MRP.
# mean price by item 
price_by_item.Item_MRP.mean()
- You can use `groupby` with **multiple** columns of the dataset too. 
- In this case, if you want to group first based on the Item_Type and then Item_MRP you can simply pass a list of column names.
# group on multiple columns
multiple_groups = data_BM[:10].groupby(['Item_Type', 'Item_Fat_Content'])
multiple_groups.first()
You can read more about **groupby** and other related functions [here.](http://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html)

#### b. **How are outlet sizes distributed based on the city type?** : crosstab

- This function is used to get an initial “feel” (view) of the data. Here, we can validate some basic hypothesis.
- For example, in this case, "Outlet_Location_Type" is expected to affect the "Outlet_Size" significantly. This can be tested using cross-tabulation as shown below:
# generate crosstab of Outlet_Size and Outlet_Location_Type
pd.crosstab(data_BM["Outlet_Size"],data_BM["Outlet_Location_Type"],margins=True)
- If you notice in the above `crosstab` there are interesting insights like 50% of medium size outlets are present only in either Tier 1 or Tier 2 cities.
- Another counter intuitive thing to notice is that high outlet size is only present in Tier 3 city though general assumption would be towards Tier 1 cities having larger outlet sizes.

#### c. How are the sales changing per year? : pivottable

- Pandas can be used to create MS Excel style pivot tables.
- The fun thing about pandas `pivot_table` is you can get another point of view on your data with only one line of code. 
- Most of the `pivot_table` parameters use default values, so the only mandatory parameters you must add are `data` and `index`. 
    - **data** is self explanatory – it’s the DataFrame you’d like to use
    - **index** is the column, grouper, array (or list of the previous) you’d like to group your data by. 
    - **values (optional)** is the column you’d like to aggregate. If you do not specify this then the function will aggregate all numeric columns.

# create pivot table
pd.pivot_table(data_BM, index=['Outlet_Establishment_Year'], values= "Item_Outlet_Sales")
- In the above example, the mean sales for each year is shown.
- You can also pass multiple columns to pivot table, in the next exammple we try to see mean sales not just by the year but also taking into account the **outlet size** and type of the city.
# create pivot table
pd.pivot_table(data_BM, index=['Outlet_Establishment_Year', 'Outlet_Location_Type', 'Outlet_Size'], values= "Item_Outlet_Sales")
- This makes it easier to see that Tier 1 cities have good sales irrespective of year and outlet size.
- We also notice that Tier 2 and Tier 3 cities dominate during the later years. This might mean both they are performing better or we have less data of later years.
- You can also perform multiple aggregations like mean, median, min, max etc. in a pivot table by using **aggfunc** parameter.
pd.pivot_table(data_BM, index=['Outlet_Establishment_Year', 'Outlet_Location_Type', 'Outlet_Size'], values= "Item_Outlet_Sales", aggfunc= [np.mean, np.median, min, max, np.std])

In [ ]:
# Data Manipulation with Pandas

Pandas is the most widely used library of python for data science. It is incredibly helpful in manipulating the data so that you can derive better insights and build great machine learning models.

In this notebook, we will have a look at some of the intermediate concepts of working with pandas.


## Table of Contents

1. Sorting dataframes
2. Merging dataframes

### Loading dataset

***In this notebook we will use the Big Mart Sales Data. You can download the data from : https://datahack.analyticsvidhya.com/contest/practice-problem-big-mart-sales-iii/download/train-file*** 
import pandas as pd
import numpy as np

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")
# view the top results
data_BM.head()
### 1. Sorting dataframes

Pandas data frame has two useful functions

- **sort_values()**: to sort pandas data frame by one or more columns
- **sort_index()**: to sort pandas data frame by row index

Each of these functions come with numerous options, like sorting the data frame in specific order (ascending or descending), sorting in place, sorting with missing values, sorting by specific algorithm etc.

Suppose you want to sort the dataframe by "Outlet_Establishment_Year" then you will use **sort_values**
# sort by year
sorted_data = data_BM.sort_values(by='Outlet_Establishment_Year')
# print sorted data
sorted_data[:5]
- Now `sort_values` takes multiple options like:
    - `ascending`: The default sorting order is ascending, when you pass False here then it sorts in descending order.
    - `inplace`: whether to do inplace sorting or not
# sort in place and descending order
data_BM.sort_values(by='Outlet_Establishment_Year', ascending=False, inplace=True)
data_BM[:5]
You might want to sort a data frame based on the values of multiple columns. 
We can specify the columns we want to sort by as a list in the argument for sort_values().

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")

# sort by multiple columns
data_BM.sort_values(by=['Outlet_Establishment_Year', 'Item_Outlet_Sales'], ascending=False)[:5]
- Note that when sorting by multiple columns, pandas sort_value() uses the first variable first and second variable next. 
- We can see the difference by switching the order of column names in the list.
# changed the order of columns
data_BM.sort_values(by=['Item_Outlet_Sales', 'Outlet_Establishment_Year'], ascending=False, inplace=True)
data_BM[:5]
- We can use **sort_index()** to sort pandas dataframe to sort by row index or names. 
- In this example, row index are numbers and in the earlier example we sorted data frame by 'Item_Outlet_Sales', 'Outlet_Establishment_Year' and therefore the row index are jumbled up. 
- We can sort by row index (with inplace=True option) and retrieve the original dataframe.
# sort by index
data_BM.sort_index(inplace=True)
data_BM[:5]
### 2. Merging dataframes

- Joining and merging DataFrames is the core process to start with data analysis and machine learning tasks. 
- It is one of the toolkits which every Data Analyst or Data Scientist should master because in almost all the cases data comes from multiple source and files.
- Pandas has two useful functions for merging dataframes:
    - **concat()**
    - **merge()** 
    
#### Creating dummy data
# create dummy data
df1 = pd.DataFrame({'A': ['A0', 'A1', 'A2', 'A3'],
                     'B': ['B0', 'B1', 'B2', 'B3'],
                     'C': ['C0', 'C1', 'C2', 'C3'],
                     'D': ['D0', 'D1', 'D2', 'D3']},
                    index=[0, 1, 2, 3])
 

df2 = pd.DataFrame({'A': ['A4', 'A5', 'A6', 'A7'],
                     'B': ['B4', 'B5', 'B6', 'B7'],
                     'C': ['C4', 'C5', 'C6', 'C7'],
                     'D': ['D4', 'D5', 'D6', 'D7']},
                    index=[4, 5, 6, 7])
 

df3 = pd.DataFrame({'A': ['A8', 'A9', 'A10', 'A11'],
                     'B': ['B8', 'B9', 'B10', 'B11'],
                     'C': ['C8', 'C9', 'C10', 'C11'],
                     'D': ['D8', 'D9', 'D10', 'D11']},
                    index=[8, 9, 10, 11])
### a. concat() for combining dataframes
- Suppose you have the following three dataframes: df1, df2 and df3 and you want to combine them **"row-wise"** so that they become a single dataframe like the given image:
![](concat_1_a.png)
- You can use **concat()** here. You will have to pass the names of the DataFrames in a list as the argument to the concat(). 
# combine dataframes
result = pd.concat([df1, df2, df3])
result
- pandas also provides you with an option to label the DataFrames, after the concatenation, with a key so that you may know which data came from which DataFrame.
- You can achieve the same by passing additional argument **keys** specifying the label names of the DataFrames in a list.
# combine dataframes
result = pd.concat([df1, df2, df3], keys=['x', 'y', 'z'])
result
- Mentioning the keys also makes it easy to retrieve data corresponding to a particular DataFrame. 
- You can retrieve the data of DataFrame df2 which had the label `y` by using the `loc` method.
# get second dataframe
result.loc['y']
- When gluing together multiple DataFrames, you have a choice of how to handle the other axes (other than the one being concatenated). This can be done in the following three ways:

    - Take the union of them all, `join='outer'`. This is the default option as it results in zero information loss.
    - Take the intersection, `join='inner'`.
    - Use a specific index, as passed to the `join_axes` argument.

- Here is an example of each of these methods. First, the default `join='outer'` behavior:
![](concat_2.png)
df4 = pd.DataFrame({'B': ['B2', 'B3', 'B6', 'B7'],
                        'D': ['D2', 'D3', 'D6', 'D7'],
                        'F': ['F2', 'F3', 'F6', 'F7']},
                       index=[2, 3, 6, 7])
    

result = pd.concat([df1, df4], axis=1, sort=False)
result
- Here is the same thing with `join='inner'`:
![](concat_3.png)
result = pd.concat([df1, df4], axis=1, join='inner')
result
- Lastly, suppose we just wanted to `reuse the exact index` from the original DataFrame:
![](concat_4.png)
result = pd.concat([df1, df4], axis=1, join_axes=[df1.index])
result
### b. merge() for combining dataframes using SQL like joins

- Another ubiquitous operation related to DataFrames is the merging operation. 
- Two DataFrames might hold different kinds of information about the same entity and linked by some common feature/column.
- We can use **merge()** to combine such dataframes in pandas.

#### Creating dummy data
# create dummy data
df_a = pd.DataFrame({
        'subject_id': ['1', '2', '3', '4', '5'],
        'first_name': ['Alex', 'Amy', 'Allen', 'Alice', 'Ayoung'], 
        'last_name': ['Anderson', 'Ackerman', 'Ali', 'Aoni', 'Atiches']})

df_b = pd.DataFrame({
        'subject_id': ['4', '5', '6', '7', '8'],
        'first_name': ['Billy', 'Brian', 'Bran', 'Bryce', 'Betty'], 
        'last_name': ['Bonder', 'Black', 'Balwner', 'Brice', 'Btisan']})

df_c = pd.DataFrame({
        'subject_id': ['1', '2', '3', '4', '5', '7', '8', '9', '10', '11'],
        'test_id': [51, 15, 15, 61, 16, 14, 15, 1, 61, 16]})
Now these are our dataframes:
![](merge_4.png)

- Let's start with a basic join, we want to combine `df_a` with `df_c` based on the `subject_id` column.
pd.merge(df_a, df_c, on='subject_id')
- Now that we have done a basic join, let's get into **some commmon SQL joins.**

#### Merge with outer join

- “Full outer join produces the set of all records in Table A and Table B, with matching records from both sides where available. If there is no match, the missing side will contain null.”
![](merge_8.png)

pd.merge(df_a, df_b, on='subject_id', how='outer')
#### Merge with inner join

- “Inner join produces only the set of records that match in both Table A and Table B.”
![](merge_5.png)
pd.merge(df_a, df_b, on='subject_id', how='inner')
#### Merge with right join

- “Right outer join produces a complete set of records from Table B, with the matching records (where available) in Table A. If there is no match, the left side will contain null.”

![](merge_7.png)
pd.merge(df_a, df_b, on='subject_id', how='right')
#### Merge with left join

- “Left outer join produces a complete set of records from Table A, with the matching records (where available) in Table B. If there is no match, the right side will contain null.”
![](merge_6.png)
pd.merge(df_a, df_b, on='subject_id', how='left')
### Merge OR Concat : Which to use when?

1. After learning both of the functions in detail, chances are that you might be confused which to use when. 
2. One major difference is that `merge()` is used to combine dataframes on the basis of values of **common columns**. While`concat()` is used to **append dataframes** one below the other (or sideways, depending on whether the axis option is set to 0 or 1).
3. Exact usage depends upon the kind of data you have and analysis you want to perform.

In [ ]:
# Data Manipulation with Pandas

Pandas is the most widely used library of python for data science. It is incredibly helpful in manipulating the data so that you can derive better insights and build great machine learning models.

In this notebook, we will have a look at some of the intermediate concepts of working with pandas.


## Table of Contents

1. Sorting dataframes
2. Merging dataframes

### Loading dataset

***In this notebook we will use the Big Mart Sales Data. You can download the data from : https://datahack.analyticsvidhya.com/contest/practice-problem-big-mart-sales-iii/download/train-file*** 
import pandas as pd
import numpy as np

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")
# view the top results
data_BM.head()
### 1. Sorting dataframes

Pandas data frame has two useful functions

- **sort_values()**: to sort pandas data frame by one or more columns
- **sort_index()**: to sort pandas data frame by row index

Each of these functions come with numerous options, like sorting the data frame in specific order (ascending or descending), sorting in place, sorting with missing values, sorting by specific algorithm etc.

Suppose you want to sort the dataframe by "Outlet_Establishment_Year" then you will use **sort_values**
# sort by year
sorted_data = data_BM.sort_values(by='Outlet_Establishment_Year')
# print sorted data
sorted_data[:5]
- Now `sort_values` takes multiple options like:
    - `ascending`: The default sorting order is ascending, when you pass False here then it sorts in descending order.
    - `inplace`: whether to do inplace sorting or not
# sort in place and descending order
data_BM.sort_values(by='Outlet_Establishment_Year', ascending=False, inplace=True)
data_BM[:5]
You might want to sort a data frame based on the values of multiple columns. 
We can specify the columns we want to sort by as a list in the argument for sort_values().

# read the dataset
data_BM = pd.read_csv('bigmart_data.csv')
# drop the null values
data_BM = data_BM.dropna(how="any")

# sort by multiple columns
data_BM.sort_values(by=['Outlet_Establishment_Year', 'Item_Outlet_Sales'], ascending=False)[:5]
- Note that when sorting by multiple columns, pandas sort_value() uses the first variable first and second variable next. 
- We can see the difference by switching the order of column names in the list.
# changed the order of columns
data_BM.sort_values(by=['Item_Outlet_Sales', 'Outlet_Establishment_Year'], ascending=False, inplace=True)
data_BM[:5]
- We can use **sort_index()** to sort pandas dataframe to sort by row index or names. 
- In this example, row index are numbers and in the earlier example we sorted data frame by 'Item_Outlet_Sales', 'Outlet_Establishment_Year' and therefore the row index are jumbled up. 
- We can sort by row index (with inplace=True option) and retrieve the original dataframe.
# sort by index
data_BM.sort_index(inplace=True)
data_BM[:5]
### 2. Merging dataframes

- Joining and merging DataFrames is the core process to start with data analysis and machine learning tasks. 
- It is one of the toolkits which every Data Analyst or Data Scientist should master because in almost all the cases data comes from multiple source and files.
- Pandas has two useful functions for merging dataframes:
    - **concat()**
    - **merge()** 
    
#### Creating dummy data
# create dummy data
df1 = pd.DataFrame({'A': ['A0', 'A1', 'A2', 'A3'],
                     'B': ['B0', 'B1', 'B2', 'B3'],
                     'C': ['C0', 'C1', 'C2', 'C3'],
                     'D': ['D0', 'D1', 'D2', 'D3']},
                    index=[0, 1, 2, 3])
 

df2 = pd.DataFrame({'A': ['A4', 'A5', 'A6', 'A7'],
                     'B': ['B4', 'B5', 'B6', 'B7'],
                     'C': ['C4', 'C5', 'C6', 'C7'],
                     'D': ['D4', 'D5', 'D6', 'D7']},
                    index=[4, 5, 6, 7])
 

df3 = pd.DataFrame({'A': ['A8', 'A9', 'A10', 'A11'],
                     'B': ['B8', 'B9', 'B10', 'B11'],
                     'C': ['C8', 'C9', 'C10', 'C11'],
                     'D': ['D8', 'D9', 'D10', 'D11']},
                    index=[8, 9, 10, 11])
### a. concat() for combining dataframes
- Suppose you have the following three dataframes: df1, df2 and df3 and you want to combine them **"row-wise"** so that they become a single dataframe like the given image:
![](concat_1_a.png)
- You can use **concat()** here. You will have to pass the names of the DataFrames in a list as the argument to the concat(). 
# combine dataframes
result = pd.concat([df1, df2, df3])
result
- pandas also provides you with an option to label the DataFrames, after the concatenation, with a key so that you may know which data came from which DataFrame.
- You can achieve the same by passing additional argument **keys** specifying the label names of the DataFrames in a list.
# combine dataframes
result = pd.concat([df1, df2, df3], keys=['x', 'y', 'z'])
result
- Mentioning the keys also makes it easy to retrieve data corresponding to a particular DataFrame. 
- You can retrieve the data of DataFrame df2 which had the label `y` by using the `loc` method.
# get second dataframe
result.loc['y']
- When gluing together multiple DataFrames, you have a choice of how to handle the other axes (other than the one being concatenated). This can be done in the following three ways:

    - Take the union of them all, `join='outer'`. This is the default option as it results in zero information loss.
    - Take the intersection, `join='inner'`.
    - Use a specific index, as passed to the `join_axes` argument.

- Here is an example of each of these methods. First, the default `join='outer'` behavior:
![](concat_2.png)
df4 = pd.DataFrame({'B': ['B2', 'B3', 'B6', 'B7'],
                        'D': ['D2', 'D3', 'D6', 'D7'],
                        'F': ['F2', 'F3', 'F6', 'F7']},
                       index=[2, 3, 6, 7])
    

result = pd.concat([df1, df4], axis=1, sort=False)
result
- Here is the same thing with `join='inner'`:
![](concat_3.png)
result = pd.concat([df1, df4], axis=1, join='inner')
result
- Lastly, suppose we just wanted to `reuse the exact index` from the original DataFrame:
![](concat_4.png)
result = pd.concat([df1, df4], axis=1, join_axes=[df1.index])
result
### b. merge() for combining dataframes using SQL like joins

- Another ubiquitous operation related to DataFrames is the merging operation. 
- Two DataFrames might hold different kinds of information about the same entity and linked by some common feature/column.
- We can use **merge()** to combine such dataframes in pandas.

#### Creating dummy data
# create dummy data
df_a = pd.DataFrame({
        'subject_id': ['1', '2', '3', '4', '5'],
        'first_name': ['Alex', 'Amy', 'Allen', 'Alice', 'Ayoung'], 
        'last_name': ['Anderson', 'Ackerman', 'Ali', 'Aoni', 'Atiches']})

df_b = pd.DataFrame({
        'subject_id': ['4', '5', '6', '7', '8'],
        'first_name': ['Billy', 'Brian', 'Bran', 'Bryce', 'Betty'], 
        'last_name': ['Bonder', 'Black', 'Balwner', 'Brice', 'Btisan']})

df_c = pd.DataFrame({
        'subject_id': ['1', '2', '3', '4', '5', '7', '8', '9', '10', '11'],
        'test_id': [51, 15, 15, 61, 16, 14, 15, 1, 61, 16]})
Now these are our dataframes:
![](merge_4.png)

- Let's start with a basic join, we want to combine `df_a` with `df_c` based on the `subject_id` column.
pd.merge(df_a, df_c, on='subject_id')
- Now that we have done a basic join, let's get into **some commmon SQL joins.**

#### Merge with outer join

- “Full outer join produces the set of all records in Table A and Table B, with matching records from both sides where available. If there is no match, the missing side will contain null.”
![](merge_8.png)

pd.merge(df_a, df_b, on='subject_id', how='outer')
#### Merge with inner join

- “Inner join produces only the set of records that match in both Table A and Table B.”
![](merge_5.png)
pd.merge(df_a, df_b, on='subject_id', how='inner')
#### Merge with right join

- “Right outer join produces a complete set of records from Table B, with the matching records (where available) in Table A. If there is no match, the left side will contain null.”

![](merge_7.png)
pd.merge(df_a, df_b, on='subject_id', how='right')
#### Merge with left join

- “Left outer join produces a complete set of records from Table A, with the matching records (where available) in Table B. If there is no match, the right side will contain null.”
![](merge_6.png)
pd.merge(df_a, df_b, on='subject_id', how='left')
### Merge OR Concat : Which to use when?

1. After learning both of the functions in detail, chances are that you might be confused which to use when. 
2. One major difference is that `merge()` is used to combine dataframes on the basis of values of **common columns**. While`concat()` is used to **append dataframes** one below the other (or sideways, depending on whether the axis option is set to 0 or 1).
3. Exact usage depends upon the kind of data you have and analysis you want to perform.